# Recursive summarization — revised 10-generation pilot

This notebook runs the revised pilot on the first **3 annotated news articles** (set `PILOT_SIZE = 100` later for the full experiment). Put the notebook in your repository root, the article JSON files in `data/`, and API keys in a local `.env` file.

**Experiment design**
- **T1 shared:** summarize the original article with a **200-word cap**.
- **Branch A, T2–T10:** progressively reduce the cap from 182 words to **40 words** at T10.
- **Branch B, T2–T10:** use a constant **200-word cap**. Both branches start from the exact same T1 summary.
- Keep the summarizer model, temperature, and common summarization prompt the same across branches; only the word-limit schedule differs.
- **Deduplicate judging, not summarization:** generate every summary normally. If the **exact summary text** has already been judged for the same article, reference facts, judge model, and prompt version, save a lightweight `is_duplicate = true` record pointing to the earlier judgment (`duplicate_of`). Do not call Claude again.
- The Excel export resolves duplicate pointers for per-generation analysis and visibly labels duplicate rows. The JSONL retains only **one full judgment per unique summary**.

The revised run writes to **new filenames** (`*_t10_v1.*`) so your older 15-generation outputs remain untouched. Both generation and judging are checkpointed and safe to resume. The word limits are *caps*, not guaranteed output lengths.


In [1]:
%pip install -q openai python-dotenv anthropic huggingface_hub pandas xlsxwriter

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from huggingface_hub import InferenceClient
from pathlib import Path
import hashlib
import json
from collections import Counter

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic

load_dotenv(".env")

DATA_DIR = Path("data")
PILOT_SIZE = 12  # Set to 100 only when ready for the full experiment.
RUN_ID = "t10_branch_t1_v1"
OUTPUT_FILE = Path("outputs/pilot_summaries_t10_llama_v1.jsonl")
MODEL = "llama-3.1-8b-instant"
SUMMARIZER_PROMPT_VERSION = "v1"
GENERATIONS = 10
BRANCH_POINT = 1
START_WORD_LIMIT = 200
END_WORD_LIMIT = 40
B_WORD_LIMIT = 200

SYSTEM_PROMPT = "You are a news summarization assistant."

def make_prompt(text, word_limit):
    return f"Summarize the following text in no more than {word_limit} words.\n\nText:\n{text}"

def branch_a_limit(generation):
    """Linearly interpolate the word cap: T1=200, T10=40 (round to nearest word)."""
    if not 1 <= generation <= GENERATIONS:
        raise ValueError(f"Invalid generation: {generation}")
    fraction = (generation - 1) / (GENERATIONS - 1)
    return round(START_WORD_LIMIT + fraction * (END_WORD_LIMIT - START_WORD_LIMIT))

def expected_word_limit(branch, generation):
    if branch == "shared" and generation == BRANCH_POINT:
        return START_WORD_LIMIT
    if branch == "A" and BRANCH_POINT < generation <= GENERATIONS:
        return branch_a_limit(generation)
    if branch == "B" and BRANCH_POINT < generation <= GENERATIONS:
        return B_WORD_LIMIT
    raise ValueError(f"Unexpected branch/generation: {(branch, generation)}")

def sha256_json(value):
    payload = json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def summary_key(article_id, branch, generation):
    return f"{article_id}_{branch}_T{generation:02d}"

print("Branch A limits:", [branch_a_limit(t) for t in range(1, GENERATIONS + 1)])
print("Branch B limit:", B_WORD_LIMIT)


c:\Penn\Python\PyVenV\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Branch A limits: [200, 182, 164, 147, 129, 111, 93, 76, 58, 40]
Branch B limit: 200


## Load the first three annotated articles

In [3]:
articles = []
for path in sorted(DATA_DIR.glob("*.json")):
    data = json.loads(path.read_text(encoding="utf-8"))
    if isinstance(data, list):
        articles.extend(data)
    elif isinstance(data, dict) and isinstance(data.get("articles"), list):
        articles.extend(data["articles"])
    else:
        articles.append(data)

pilot = articles[:PILOT_SIZE]
if len(pilot) < PILOT_SIZE:
    raise ValueError(f"Found {len(pilot)} articles in {DATA_DIR}; expected at least {PILOT_SIZE}.")
if len({str(a.get("article_id")) for a in pilot}) != len(pilot):
    raise ValueError("Duplicate article_id in selected articles.")

for article in pilot:
    if not all(k in article for k in ("article_id", "true_text", "atomic_facts")):
        raise ValueError("Each article needs 'article_id', 'true_text', and 'atomic_facts'.")
    print(article["article_id"], "—", article.get("title", ""))


NEWS_001 — Yousef Makki: Boy, 17, cleared of stabbing teenager to death
NEWS_002 — Whistleblowers: 'We spoke out and lost our jobs'
NEWS_003 — YouTube users can't stop streaming Latin Pop
NEWS_004 — Russia mini-sub fire: Putin issues heroism awards
NEWS_005 — Boris Johnson in 'deal or no deal' Brexit challenge to rival Hunt
NEWS_006 — Contaminated blood inquiry: More witnesses prompt extension
NEWS_007 — Wireless electric car charging gets cash boost
NEWS_008 — Wales' Jewish history: Call to record it before it is too late
NEWS_009 — Contaminated blood inquiry: Welsh Government urged to take lead
NEWS_010 — Iran nuclear deal: Uranium enrichment breaches are extortion, says US
NEWS_011 — New UK prime minister: Johnson and Hunt await Conservative leadership vote
NEWS_012 — Tory leadership: What's Jeremy Hunt's track record?


## Generate and checkpoint summaries (every generation, including repeats)

In [ ]:
client = InferenceClient(token=os.environ["HF_TOKEN"])
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

# Checkpoint every summary. Never mix old schedules, prompts, or article versions.
done = {}
if OUTPUT_FILE.exists():
    for line in OUTPUT_FILE.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        row = json.loads(line)
        key = (str(row["article_id"]), row["branch"], int(row["generation"]))
        if key in done:
            raise ValueError(f"Duplicate summary checkpoint: {key}")
        if (row.get("run_id") != RUN_ID or row.get("model") != MODEL
                or row.get("summarizer_prompt_version") != SUMMARIZER_PROMPT_VERSION
                or row.get("word_limit") != expected_word_limit(row["branch"], row["generation"])):
            raise ValueError(f"Incompatible summary checkpoint {key}. Use a new OUTPUT_FILE.")
        done[key] = row

# A source-text change must invalidate old summary checkpoints.
article_by_id = {str(a["article_id"]): a for a in pilot}
for key, row in done.items():
    if key[0] in article_by_id:
        source_hash = sha256_json(article_by_id[key[0]]["true_text"])
        if row.get("source_hash") != source_hash:
            raise ValueError(f"Source article changed for {key}. Use a new OUTPUT_FILE.")

def summarize(article, branch, generation, text, word_limit):
    key = (str(article["article_id"]), branch, generation)
    if word_limit != expected_word_limit(branch, generation):
        raise ValueError(f"Unexpected word limit for {key}: {word_limit}")
    if key in done:
        return done[key]["summary"]

    response = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        max_tokens=1024,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": make_prompt(text, word_limit)},
        ],
    )
    summary = (response.choices[0].message.content or "").strip()
    if not summary:
        raise RuntimeError(f"Empty response for {key}; nothing saved.")

    row = {
        "article_id": str(article["article_id"]),
        "title": article.get("title", ""),
        "model": MODEL,
        "run_id": RUN_ID,
        "summarizer_prompt_version": SUMMARIZER_PROMPT_VERSION,
        "source_hash": sha256_json(article["true_text"]),
        "branch": branch,
        "generation": generation,
        "word_limit": word_limit,
        "summary": summary,
        "summary_words": len(summary.split()),
    }
    with OUTPUT_FILE.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
    done[key] = row
    return summary


In [5]:
for article in pilot:
    # T1 is shared; both branches start from this exact text.
    shared_text = summarize(article, "shared", 1, article["true_text"], START_WORD_LIMIT)

    for branch in ("A", "B"):
        current = shared_text
        for t in range(BRANCH_POINT + 1, GENERATIONS + 1):
            limit = expected_word_limit(branch, t)
            current = summarize(article, branch, t, current, limit)

    print("Completed:", article["article_id"])

expected_count = len(pilot) * (BRANCH_POINT + 2 * (GENERATIONS - BRANCH_POINT))
selected_count = sum(1 for k in done if k[0] in article_by_id)
print(f"Saved {selected_count}/{expected_count} summaries for the selected articles to {OUTPUT_FILE}")


NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-3.1-8b-instant` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

**Summary output:** one JSONL row for each generation (`shared` T1; `A` and `B` T2–T10), even if the text repeats. The row includes `run_id`, `source_hash`, `summarizer_prompt_version`, and the actual word cap. Re-running the generation cells resumes from the checkpoint. Do **not** reuse the old `pilot_summaries.jsonl` from the 15-generation experiment.


## Judge unique summaries with Claude (exact-text deduplication)

Add `ANTHROPIC_API_KEY=your_key_here` to `.env` (or set it in the environment). Do not paste keys into the notebook.

For every article, Claude compares each **unique** summary against that article's unchanged original atomic-fact inventory. The labels are `present`, `absent`, `mutated`, plus unverified `candidate_hallucination` claims. The full original article is not sent to Claude.

**Exact duplicate rule:** if the summary text matches an earlier summary **byte-for-byte** within the same article and the same reference facts/judge configuration, save a lightweight record with `is_duplicate = True` and `duplicate_of = <original summary_key>`. Do not call Claude and do not repeat the fact arrays in the JSONL. This applies across branches as well as within a branch. **All generations are still present.**

`MAX_JUDGE_CALLS = 1` is a safe first test; set it to `None` to complete the pilot. A checkpoint resumes without rejudging completed summaries. Invalid exact evidence quotes produce warnings and `evidence_valid = False`, not a fatal error. The Excel export below resolves duplicate pointers to show per-generation metrics.


In [ ]:
JUDGE_MODEL = "claude-haiku-5-5"
JUDGE_PROMPT_VERSION = "v1"
JUDGE_OUTPUT = Path("outputs/pilot_judgments_t10_llama_v1.jsonl")
MAX_JUDGE_CALLS = None  # First test: 1 API call. Set to None for the full pilot.

JUDGE_PROMPT = """You are an independent factual-consistency judge for a recursive news summarization experiment.

You receive (1) a fixed list of atomic facts extracted from the ORIGINAL article and (2) ONE generated summary. Evaluate ONLY this summary. Do not use outside knowledge, previous summaries, or the full original article. The reference fact IDs must remain unchanged.

TASK A — Evaluate EVERY original fact exactly once:

- present: The summary preserves the factual meaning of the reference fact. Accept paraphrases, synonyms, reordering, and equivalent expressions. Do not require exact wording. All essential factual information must be preserved, including important numbers, entities, attribution, negation, uncertainty, and causal relationships.

- absent: The summary does not convey the reference fact, or omits the information needed to identify the particular factual claim.

- mutated: The summary conveys an identifiable version of the reference fact but changes its factual meaning, introduces an incorrect detail, or materially alters its specificity, certainty, attribution, negation, or causal relationship. Do not classify stylistic changes or semantically equivalent paraphrases as mutations.

When a summary expresses a broader but accurate version of a fact, judge whether the omitted specificity is essential. If the core claim remains intact and no material factual meaning changes, classify it as present. If essential specificity is lost, classify it as mutated.

For each reference fact, return its original fact_id, label, an EXACT quote from the SUMMARY as evidence (empty string if absent), and a brief reason for mutations (empty string otherwise). Never use the original fact's source_span as summary evidence.

TASK B — Identify candidate new claims:

- Identify substantive, independently verifiable factual claims in the summary that are neither represented by nor reasonably entailed by ANY reference fact.
- Accept semantic equivalence, paraphrases, and straightforward implications of reference facts.
- Do not flag changes in wording, summarizing language, generalizations, or reasonable contextual restatements as new claims.
- Do not duplicate a mutation as a new claim.
- Only flag a claim when it introduces a genuinely new factual assertion not supported by the reference inventory.
- Write each new claim as a short, stand-alone atomic claim with an EXACT quote from the summary.

These are CANDIDATE unsupported claims, not confirmed hallucinations. The reference inventory may omit information present in the original article.

Use only the provided summary as evidence for what the summary says. Keep all generations independent. Return the structured JSON requested by the API schema, with no additional commentary.
"""

JUDGE_SCHEMA = {
    "type": "object",
    "properties": {
        "fact_evaluations": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "fact_id": {"type": "string"},
                    "label": {"type": "string", "enum": ["present", "absent", "mutated"]},
                    "evidence": {"type": "string"},
                    "reason": {"type": "string"},
                },
                "required": ["fact_id", "label", "evidence", "reason"],
                "additionalProperties": False,
            },
        },
        "new_claims": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "fact": {"type": "string"},
                    "evidence": {"type": "string"},
                },
                "required": ["fact", "evidence"],
                "additionalProperties": False,
            },
        },
    },
    "required": ["fact_evaluations", "new_claims"],
    "additionalProperties": False,
}


In [ ]:
judge_client = Anthropic()  # Uses ANTHROPIC_API_KEY from environment / .env
JUDGE_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

article_by_id = {str(a["article_id"]): a for a in pilot}
if not OUTPUT_FILE.exists():
    raise FileNotFoundError(f"No summaries at {OUTPUT_FILE}. Run the summarization cells first.")

summary_rows = [json.loads(line) for line in OUTPUT_FILE.read_text(encoding="utf-8").splitlines() if line.strip()]
summary_rows = [row for row in summary_rows if str(row["article_id"]) in article_by_id]
branch_order = {"shared": 0, "A": 1, "B": 2}
article_order = {str(a["article_id"]): i for i, a in enumerate(pilot)}
summary_rows.sort(key=lambda row: (
    article_order[str(row["article_id"])], branch_order[row["branch"]], row["generation"]
))

expected_count = len(pilot) * (BRANCH_POINT + 2 * (GENERATIONS - BRANCH_POINT))
if len(summary_rows) != expected_count:
    raise ValueError(f"Found {len(summary_rows)}/{expected_count} summaries. Finish the generation cell first.")
summary_by_key = {}
for row in summary_rows:
    key = summary_key(str(row["article_id"]), row["branch"], int(row["generation"]))
    if key in summary_by_key:
        raise ValueError(f"Duplicate summary checkpoint: {key}")
    if (row.get("run_id") != RUN_ID or row.get("model") != MODEL
            or row.get("summarizer_prompt_version") != SUMMARIZER_PROMPT_VERSION
            or row.get("word_limit") != expected_word_limit(row["branch"], row["generation"])):
        raise ValueError(f"Summary checkpoint is from another configuration: {key}")
    if row.get("source_hash") != sha256_json(article_by_id[str(row["article_id"])]["true_text"]):
        raise ValueError(f"Source article changed for {key}")
    summary_by_key[key] = row

# One fingerprint per original reference inventory, used to prevent stale cache reuse.
reference_hashes = {
    article_id: sha256_json(article["atomic_facts"])
    for article_id, article in article_by_id.items()
}

judged = {}
if JUDGE_OUTPUT.exists():
    for line in JUDGE_OUTPUT.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        row = json.loads(line)
        key = (str(row["article_id"]), row["branch"], int(row["generation"]))
        if key in judged:
            raise ValueError(f"Duplicate judge checkpoint: {key}")
        if (row.get("judge_model") != JUDGE_MODEL
                or row.get("judge_prompt_version") != JUDGE_PROMPT_VERSION
                or row.get("run_id") != RUN_ID):
            raise ValueError(f"Incompatible judge checkpoint {key}. Use a new JUDGE_OUTPUT.")
        judged[key] = row

# Build the cache in chronological evaluation order. Values are keys of FULL judgments.
unique_by_text = {}  # (article_id, exact_summary_text, reference_hash) -> original summary_key
processed = {}       # summary_key -> saved judgment or duplicate pointer
new_calls = 0
new_duplicates = 0

for summary_row in summary_rows:
    article_id = str(summary_row["article_id"])
    key = (article_id, summary_row["branch"], int(summary_row["generation"]))
    this_summary_key = summary_key(*key)
    reference_hash = reference_hashes[article_id]
    cache_key = (article_id, summary_row["summary"], reference_hash)

    if key in judged:
        existing = judged[key]
        if (existing.get("summary") != summary_row["summary"]
                or existing.get("reference_hash") != reference_hash
                or existing.get("word_limit") != summary_row["word_limit"]):
            raise ValueError(f"Judgment checkpoint no longer matches summary/reference: {key}")
        if existing.get("is_duplicate", False):
            target = existing.get("duplicate_of")
            if target not in processed or processed[target].get("is_duplicate", False):
                raise ValueError(f"Invalid duplicate pointer for {this_summary_key}: {target}")
            if processed[target]["summary"] != summary_row["summary"]:
                raise ValueError(f"Duplicate pointer has different text for {this_summary_key}")
        else:
            if cache_key in unique_by_text:
                raise ValueError(f"Repeated full judgment instead of duplicate: {this_summary_key}")
            unique_by_text[cache_key] = this_summary_key
        processed[this_summary_key] = existing
        continue

    # Exact same text + article + reference facts + judge model/prompt (enforced by checkpoint).
    if cache_key in unique_by_text:
        duplicate_of = unique_by_text[cache_key]
        output_row = {
            **summary_row,
            "judge_model": JUDGE_MODEL,
            "judge_prompt_version": JUDGE_PROMPT_VERSION,
            "reference_hash": reference_hash,
            "is_duplicate": True,
            "duplicate_of": duplicate_of,
        }
        with JUDGE_OUTPUT.open("a", encoding="utf-8") as f:
            f.write(json.dumps(output_row, ensure_ascii=False) + "\n")
        judged[key] = output_row
        processed[this_summary_key] = output_row
        new_duplicates += 1
        print(f"Duplicate {this_summary_key} -> {duplicate_of}")
        continue

    if MAX_JUDGE_CALLS is not None and new_calls >= MAX_JUDGE_CALLS:
        break

    original_facts = article_by_id[article_id]["atomic_facts"]
    reference = [
        {k: f.get(k) for k in (
            "fact_id", "fact", "type", "key_value", "negation",
            "uncertainty", "quotation", "attribution", "opinion"
        )}
        for f in original_facts
    ]
    response = judge_client.messages.create(
        model=JUDGE_MODEL,
        max_tokens=16000,
        system=JUDGE_PROMPT,
        messages=[{"role": "user", "content": json.dumps({
            "reference_facts": reference,
            "summary": summary_row["summary"],
        }, ensure_ascii=False)}],
        output_config={"format": {"type": "json_schema", "schema": JUDGE_SCHEMA}},
    )
    if response.stop_reason != "end_turn":
        raise RuntimeError(f"Incomplete judge response for {key}: {response.stop_reason}")
    result = json.loads(next(block.text for block in response.content if block.type == "text"))

    fact_by_id = {f["fact_id"]: f for f in original_facts}
    returned_ids = [f["fact_id"] for f in result["fact_evaluations"]]
    if len(returned_ids) != len(fact_by_id) or set(returned_ids) != set(fact_by_id):
        raise ValueError(f"Missing, duplicate, or unknown fact IDs in judge response for {key}")

    for item in result["fact_evaluations"]:
        item["evidence_valid"] = (not item["evidence"] or item["evidence"] in summary_row["summary"])
        if not item["evidence_valid"]:
            print(f"Warning: Evidence mismatch for {key}, {item['fact_id']}")
        original = fact_by_id[item["fact_id"]]
        item["reference_fact"] = original["fact"]
        item["type"] = original.get("type")
        item["salience"] = original.get("salience")
        item["needs_review"] = item["label"] == "mutated"
        item["reviewed_label"] = None

    for item in result["new_claims"]:
        item["evidence_valid"] = (bool(item["evidence"]) and item["evidence"] in summary_row["summary"])
        if not item["evidence_valid"]:
            print(f"Warning: New-claim evidence mismatch for {key}")
        item["label"] = "candidate_hallucination"
        item["needs_review"] = True
        item["reviewed_label"] = None

    output_row = {
        **summary_row,
        "judge_model": JUDGE_MODEL,
        "judge_prompt_version": JUDGE_PROMPT_VERSION,
        "reference_hash": reference_hash,
        "is_duplicate": False,
        "duplicate_of": None,
        **result,
    }
    with JUDGE_OUTPUT.open("a", encoding="utf-8") as f:
        f.write(json.dumps(output_row, ensure_ascii=False) + "\n")
    judged[key] = output_row
    processed[this_summary_key] = output_row
    unique_by_text[cache_key] = this_summary_key
    new_calls += 1
    print(f"Judged {this_summary_key}: "
          f"{sum(f['label'] == 'mutated' for f in result['fact_evaluations'])} mutations, "
          f"{len(result['new_claims'])} candidate new claims")

print(f"Saved {len(judged)}/{len(summary_rows)} generation records to {JUDGE_OUTPUT}")
print(f"This run: {new_calls} Claude calls, {new_duplicates} exact duplicates referenced")


## Export judgments to Excel (duplicate-aware)

The workbook contains **Overview**, **Fact Judgments**, **New Claims**, and **Summaries**. Duplicate rows are labeled `is_duplicate = True` with a `duplicate_of` pointer. For complete per-generation trajectories, the export **resolves that pointer in memory** and shows the source judgment's effective labels/counts under the duplicate generation. It does **not** write copied fact arrays to the JSONL or call either API.

For analyses that require independent judge responses, filter `is_duplicate = False`. For trajectories over all ten generations, keep both judged and duplicate rows. `candidate_hallucination` means a claim absent from the reference-fact inventory, not a confirmed falsehood. Rerun this cell after additional judgments are checkpointed.


In [ ]:
# Excel is a readable, duplicate-aware view of the checkpointed judgments.
JUDGE_JSONL = JUDGE_OUTPUT
EXCEL_OUTPUT = Path("outputs/pilot_judgments_t10_llama_v1.xlsx")

if not JUDGE_JSONL.exists():
    raise FileNotFoundError(f"Run the Claude judge first: {JUDGE_JSONL}")

judgments = [json.loads(line) for line in JUDGE_JSONL.read_text(encoding="utf-8").splitlines() if line.strip()]
if not judgments:
    raise ValueError(f"No judgments found in {JUDGE_JSONL}")

branch_order = {"shared": 0, "A": 1, "B": 2}
judgments.sort(key=lambda r: (str(r["article_id"]), branch_order.get(r["branch"], 9), int(r["generation"])))

judgments_by_key = {}
for row in judgments:
    key = summary_key(str(row["article_id"]), row["branch"], int(row["generation"]))
    if key in judgments_by_key:
        raise ValueError(f"Duplicate checkpoint for {key}; check {JUDGE_JSONL}")
    if row.get("run_id") != RUN_ID or row.get("judge_model") != JUDGE_MODEL:
        raise ValueError(f"Unexpected run/model in {key}; check {JUDGE_JSONL}")
    judgments_by_key[key] = row

fact_rows, claim_rows, summary_export_rows, overview_rows = [], [], [], []
for row in judgments:
    article_id, branch, generation = str(row["article_id"]), row["branch"], int(row["generation"])
    key = summary_key(article_id, branch, generation)
    is_duplicate = bool(row.get("is_duplicate", False))
    duplicate_of = row.get("duplicate_of") if is_duplicate else None
    source = row
    if is_duplicate:
        source = judgments_by_key.get(duplicate_of)
        if source is None or source.get("is_duplicate", False):
            raise ValueError(f"Invalid duplicate_of for {key}: {duplicate_of}")
        if (source["article_id"] != article_id
                or source["summary"] != row["summary"]
                or source.get("reference_hash") != row.get("reference_hash")
                or source.get("judge_prompt_version") != row.get("judge_prompt_version")):
            raise ValueError(f"Duplicate pointer mismatches source for {key}")

    base = {
        "summary_key": key,
        "article_id": article_id,
        "branch": branch,
        "generation": generation,
        "is_duplicate": is_duplicate,
        "duplicate_of": duplicate_of or "",
    }
    evaluations = source["fact_evaluations"]
    claims = source.get("new_claims", [])
    labels = Counter(f["label"] for f in evaluations)

    for f in evaluations:
        fact_rows.append({
            **base,
            "fact_id": f["fact_id"],
            "type": f.get("type", ""),
            "salience": f.get("salience", ""),
            "reference_fact": f.get("reference_fact", ""),
            "claude_label": f["label"],
            "summary_evidence": f.get("evidence", ""),
            "evidence_valid": f.get("evidence_valid", ""),
            "mutation_reason": f.get("reason", ""),
        })
    for i, claim in enumerate(claims, start=1):
        claim_rows.append({
            **base,
            "claim_id": f"{key}_N{i:02d}",
            "claim": claim["fact"],
            "summary_evidence": claim.get("evidence", ""),
            "evidence_valid": claim.get("evidence_valid", ""),
            "claude_label": "candidate_hallucination",
        })

    summary_export_rows.append({
        **base,
        "title": row.get("title", ""),
        "summary_words": row.get("summary_words"),
        "word_limit": row.get("word_limit"),
        "summary": row["summary"],
        "summarizer_model": row.get("model", ""),
        "judge_model": row.get("judge_model", ""),
        "judge_prompt_version": row.get("judge_prompt_version", ""),
    })
    overview_rows.append({
        **base,
        "title": row.get("title", ""),
        "reference_facts": len(evaluations),
        "present": labels["present"],
        "absent": labels["absent"],
        "mutated": labels["mutated"],
        "candidate_hallucinations": len(claims),
        "summary_words": row.get("summary_words"),
    })

sheets = {
    "Overview": pd.DataFrame(overview_rows),
    "Fact Judgments": pd.DataFrame(fact_rows),
    "New Claims": pd.DataFrame(claim_rows, columns=[
        "summary_key", "article_id", "branch", "generation", "is_duplicate", "duplicate_of",
        "claim_id", "claim", "summary_evidence", "evidence_valid", "claude_label"
    ]),
    "Summaries": pd.DataFrame(summary_export_rows),
}

EXCEL_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(EXCEL_OUTPUT, engine="xlsxwriter", engine_kwargs={
    "options": {"strings_to_formulas": False, "strings_to_urls": False}
}) as writer:
    workbook = writer.book
    header_fmt = workbook.add_format({
        "bold": True, "font_color": "#FFFFFF", "bg_color": "#263F60",
        "valign": "vcenter", "text_wrap": True, "bottom": 1, "bottom_color": "#CFD8E3"
    })
    wrap_fmt = workbook.add_format({"text_wrap": True, "valign": "top"})
    label_colors = {
        "present": "#E2F2E8",
        "absent": "#E9EDF2",
        "mutated": "#FFF0D2",
        "candidate_hallucination": "#F1E8FA",
    }
    for sheet_name, df in sheets.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)
        sheet = writer.sheets[sheet_name]
        sheet.freeze_panes(1, 1)
        sheet.set_row(0, 30, header_fmt)
        sheet.set_default_row(21)
        if len(df):
            sheet.autofilter(0, 0, len(df), len(df.columns) - 1)
        for i, col in enumerate(df.columns):
            if col == "summary":
                width = 85
            elif col in ("reference_fact", "claim", "summary_evidence", "mutation_reason"):
                width = 56
            elif col == "title":
                width = 44
            elif col in ("summary_key", "claim_id", "duplicate_of"):
                width = 30
            elif col in ("claude_label", "candidate_hallucinations"):
                width = 25
            elif col in ("fact_id", "judge_model", "summarizer_model"):
                width = 24
            else:
                width = 16
            sheet.set_column(i, i, width, wrap_fmt if col in (
                "reference_fact", "claim", "summary_evidence", "mutation_reason", "summary", "title"
            ) else None)
        if len(df) and "claude_label" in df.columns:
            label_col = df.columns.get_loc("claude_label")
            for label, color in label_colors.items():
                fmt = workbook.add_format({"bg_color": color})
                sheet.conditional_format(1, label_col, len(df), label_col, {
                    "type": "cell", "criteria": "==", "value": f'"{label}"', "format": fmt
                })
        if sheet_name == "Summaries":
            sheet.set_default_row(125)
        elif sheet_name in ("Fact Judgments", "New Claims"):
            sheet.set_default_row(40)

print(f"Exported {len(judgments)} generation records to {EXCEL_OUTPUT}")
print(f"  {sum(bool(r.get('is_duplicate')) for r in judgments)} duplicate references; "
      f"{len(fact_rows)} per-generation fact rows; {len(claim_rows)} candidate claim rows")
print("Duplicate counts/labels are resolved from the original judgment; no duplicate API calls.")
